# Generador v2 de LetraDeMédico: QLoRA con el dataset limpio (Colab T4)

Mismo modelo y mismos hiperparámetros que el v1 (Qwen2.5-3B-Instruct, 4 bits, LoRA r = 16, lr 2e-4, 2 épocas,
lote efectivo 16). **Solo cambian los datos:** `sft_train_v2.jsonl` (421 ejemplos) en lugar de los 555 del v1.
El dataset v2 quita los ejemplos que:
- no mencionan el valor de un examen fuera de rango;
- copian la misma frase de causas en varios exámenes;
- inventan el significado de una sigla;
- usan "usted" en vez de "tú".

Además limpia las preguntas con la misma revisión de la app. Ver `training/generador/dataset.py`.

Antes de abrirlo, el dataset v2 tiene que estar en GitHub (`git push`), porque este notebook clona el repositorio.

Tiempo aproximado en la T4:
1. Entrenar: unos 40 minutos.
2. Responder los 150 casos de test: unos 35 minutos.
3. Exportar a GGUF: unos 20 minutos.

Los checkpoints, las respuestas y el GGUF se guardan en Google Drive: si Colab se desconecta, se vuelve a correr
desde la celda 1 y cada paso continúa donde iba.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/letrademedico'
ADAPTADOR = f'{DRIVE}/qwen2.5-3b-instruct-lora-v2'
GGUF = f'{DRIVE}/letrademedico-qwen2.5-3b-v2-q4_k_m.gguf'
!mkdir -p {DRIVE}/checkpoints-v2 {DRIVE}/salidas

In [ ]:
%cd /content
!test -d Proyecto_LaLetraDeMedico || git clone https://github.com/Valero25/Proyecto_LaLetraDeMedico.git
%cd /content/Proyecto_LaLetraDeMedico
!git pull -q && git log --oneline -1
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets
# Las salidas del test van directo a Drive, para no perderlas si se corta la sesión
!rm -rf data/generador/salidas && ln -s {DRIVE}/salidas data/generador/salidas
!wc -l data/generador/sft_train_v2.jsonl data/generador/sft_val_v2.jsonl

## 1. Entrenar el v2 (se reanuda desde el último checkpoint si Colab se desconectó)

In [ ]:
%cd /content/Proyecto_LaLetraDeMedico
!python training/generador/ajustar_qlora.py --version v2 --checkpoints {DRIVE}/checkpoints-v2 --salida {ADAPTADOR}
!cat {ADAPTADOR}/entrenamiento.json | head -c 600

## 2. Responder los 150 casos de test (los mismos del v1; reanudable)

In [ ]:
%cd /content/Proyecto_LaLetraDeMedico
!python training/generador/generar_respuestas.py --adaptador {ADAPTADOR} --nombre qwen2.5-3b-ajustado-v2
!wc -l {DRIVE}/salidas/qwen2.5-3b-ajustado-v2.jsonl

## 3. Exportar a GGUF Q4_K_M (como en `exportar_gguf_colab.ipynb`)

Fusiona el LoRA con el modelo base en float16 (en CPU) y cuantiza a 4 bits para llama.cpp.

In [ ]:
# Colab trae torchao 0.10, que la versión actual de peft rechaza; no se usa, así que se quita
!pip uninstall -y -q torchao
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct', dtype=torch.float16, device_map='cpu')
fusionado = PeftModel.from_pretrained(base, ADAPTADOR).merge_and_unload()
fusionado.save_pretrained('/content/fusionado-v2', safe_serialization=True)
AutoTokenizer.from_pretrained(ADAPTADOR).save_pretrained('/content/fusionado-v2')
del base, fusionado

In [ ]:
%cd /content
!test -d llama.cpp || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q gguf sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py /content/fusionado-v2 --outfile /content/ldm-v2-f16.gguf --outtype f16
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!llama.cpp/build/bin/llama-quantize /content/ldm-v2-f16.gguf "{GGUF}" Q4_K_M
!ls -lh "{GGUF}"

## 4. Descargar el adaptador y las respuestas del test (pequeños)

In [ ]:
!rm -rf /content/v2 && mkdir -p /content/v2/models/generador /content/v2/data/generador/salidas
!cp -r {ADAPTADOR} /content/v2/models/generador/
!cp {DRIVE}/salidas/qwen2.5-3b-ajustado-v2.jsonl /content/v2/data/generador/salidas/
%cd /content/v2
!zip -qr /content/generador_v2.zip models data && ls -lh /content/generador_v2.zip
from google.colab import files
files.download('/content/generador_v2.zip')

## 5. En el computador local

1. Descomprime `generador_v2.zip` en la raíz del proyecto. Deja `models/generador/qwen2.5-3b-instruct-lora-v2/` y `data/generador/salidas/qwen2.5-3b-ajustado-v2.jsonl`.
2. Descarga desde Google Drive (`MyDrive/letrademedico/`) el archivo `letrademedico-qwen2.5-3b-v2-q4_k_m.gguf`, de unos 2 GB, y cópialo a `models/generador/`. La descarga directa desde Colab suele fallar con archivos así de grandes.
3. Evalúa v1 frente a v2 sobre el mismo test: `python evaluacion/evaluar_generador.py`.
4. Para usar el v2 en la app: `$env:LDM_MODELO_GENERADOR = "models/generador/letrademedico-qwen2.5-3b-v2-q4_k_m.gguf"` y luego `streamlit run app.py`.